In [1]:
import importlib

import pandas as pd

import attrition_analysis.data_selection as data_selection
import attrition_analysis.models_utils as models_utils

importlib.reload(models_utils)
importlib.reload(data_selection)

from attrition_analysis.data_selection import CATEGORICAL_MODEL_VARS, MIXED_MODEL_VARS
from attrition_analysis.models_utils import (
    estimators_dict,
    evaluate_thresholds_optimized_models,
    mixed_models_dict_c,
    categorical_models_dict_c,
    param_distributions_dict,
    run_cross_validation_mixed,
    run_model_comparison_mixed,
    tune_hyperparameters_top_combinations,
)

df = pd.read_csv("../../data/clean/Employee-Attrition_Clean.csv")

target = "AttritionFlag"

df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,Gender,...,IncomeGroup,EducationLevel,StockOption,JobLevelGroup,SatisfactionLevel,PerformanceRatingLevel,EnvironmentSatisfactionLevel,JobInvolvementLevel,RelationshipSatisfactionLevel,WorkLifeBalanceLevel
0,41,Yes,Travel_Rarely,1102,Sales,1,Life Sciences,1,1,Female,...,Medium,College,No Options,Junior Level,Very High,Excellent,Medium,High,Low,Bad
1,49,No,Travel_Frequently,279,Research & Development,8,Life Sciences,1,2,Male,...,Medium,Below College,Low,Junior Level,Medium,Outstanding,High,Medium,Very High,Better
2,37,Yes,Travel_Rarely,1373,Research & Development,2,Other,1,4,Male,...,Low,College,No Options,Entry Level,High,Excellent,Very High,Medium,Medium,Better
3,33,No,Travel_Frequently,1392,Research & Development,3,Life Sciences,1,5,Female,...,Low,Master,No Options,Entry Level,High,Excellent,Very High,High,High,Better
4,27,No,Travel_Rarely,591,Research & Development,2,Medical,1,7,Male,...,Medium,Below College,Low,Entry Level,Medium,Excellent,Low,High,Very High,Better
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,Medical,1,2061,Male,...,Low,College,Low,Junior Level,Very High,Excellent,High,Very High,High,Better
1466,39,No,Travel_Rarely,613,Research & Development,6,Medical,1,2062,Male,...,High,Below College,Low,Mid Level,Low,Excellent,Very High,Medium,Low,Better
1467,27,No,Travel_Rarely,155,Research & Development,4,Life Sciences,1,2064,Male,...,High,Bachelor,Low,Junior Level,Medium,Outstanding,Medium,Very High,Medium,Better
1468,49,No,Travel_Frequently,1023,Sales,2,Medical,1,2065,Male,...,Medium,Bachelor,No Options,Junior Level,Medium,Excellent,Very High,Medium,Very High,Good


In [2]:
all_models_dict_c = {**categorical_models_dict_c, **mixed_models_dict_c}

# Comparação dos Modelos Categóricos

In [3]:
general_comparison_c, threshold_comparison_c, confusion_results_c, trained_models_c, interpretation_results_c = run_model_comparison_mixed(
    df=df,
    models_dict=categorical_models_dict_c,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

general_comparison_c.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
45,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.5,0.780,0.402,0.746,0.522,0.798,0,9,24
11,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.5,0.773,0.393,0.746,0.515,0.818,0,8,22
27,Modelo 3 — Faixa Salarial,Gradient Boosting Balanced,0.5,0.785,0.400,0.676,0.503,0.799,0,8,21
49,Modelo 5 — Estabilidade e Benefícios,XGBoost Balanced,0.5,0.780,0.393,0.676,0.497,0.792,0,9,24
57,Modelo 6 — Perfil Pessoal,Gradient Boosting Balanced,0.5,0.794,0.407,0.620,0.492,0.795,0,9,24
...,...,...,...,...,...,...,...,...,...,...,...
54,Modelo 6 — Perfil Pessoal,Random Forest,0.5,0.844,1.000,0.028,0.055,0.793,0,9,24
4,Modelo 1 — Função Profissional,Random Forest,0.5,0.841,1.000,0.014,0.028,0.768,0,8,26
64,Modelo 7 — Reduzido Conservador,Random Forest,0.5,0.839,0.000,0.000,0.000,0.755,0,7,18
44,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.5,0.839,0.000,0.000,0.000,0.802,0,9,24


In [4]:
c_best_thresholds_f1 = threshold_comparison_c.loc[
    threshold_comparison_c.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

c_best_thresholds_f1.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
25,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.841,0.506,0.592,0.545,0.795
15,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.575,0.819,0.456,0.662,0.540,0.818
14,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.810,0.441,0.690,0.538,0.819
19,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.650,0.844,0.513,0.563,0.537,0.793
47,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.525,0.807,0.438,0.690,0.536,0.798
...,...,...,...,...,...,...,...,...
51,Modelo 6 — Perfil Pessoal,Decision Tree Balanced,0.650,0.830,0.468,0.408,0.436,0.712
60,Modelo 7 — Reduzido Conservador,Decision Tree,0.225,0.739,0.333,0.620,0.433,0.705
61,Modelo 7 — Reduzido Conservador,Decision Tree Balanced,0.425,0.705,0.310,0.676,0.425,0.728
1,Modelo 1 — Função Profissional,Decision Tree Balanced,0.650,0.800,0.392,0.437,0.413,0.720


In [5]:
c_best_thresholds_by_model = threshold_comparison_c.loc[
    threshold_comparison_c.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

c_best_thresholds_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
5,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.841,0.506,0.592,0.545,0.795
4,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.810,0.441,0.690,0.538,0.819
9,Modelo 2 — Nível Hierárquico,XGBoost Balanced,0.650,0.844,0.513,0.563,0.537,0.793
7,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.525,0.807,0.438,0.690,0.536,0.798
2,Modelo 3 — Faixa Salarial,Gradient Boosting,0.275,0.844,0.513,0.549,0.531,0.797
8,Modelo 2 — Nível Hierárquico,XGBoost,0.300,0.846,0.521,0.521,0.521,0.804
3,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.650,0.839,0.500,0.535,0.517,0.805
6,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.200,0.785,0.402,0.690,0.508,0.802
0,Modelo 5 — Estabilidade e Benefícios,Decision Tree,0.275,0.821,0.452,0.535,0.490,0.768
1,Modelo 3 — Faixa Salarial,Decision Tree Balanced,0.600,0.816,0.439,0.507,0.471,0.778


In [6]:
c_threshold_mean = threshold_comparison_c.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean().sort_values(by="F1-score", ascending=False)

c_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 3 — Faixa Salarial,0.751916,0.454121,0.534874,0.404879,0.7874
Modelo 2 — Nível Hierárquico,0.742158,0.438958,0.527747,0.384132,0.7901
Modelo 6 — Perfil Pessoal,0.756516,0.435084,0.501426,0.382642,0.7898
Modelo 5 — Estabilidade e Benefícios,0.750211,0.439300,0.508663,0.371595,0.7893
Modelo 4 — Trajetória Organizacional,0.745668,0.400258,0.487100,0.367795,0.7583
Modelo 1 — Função Profissional,0.748132,0.439758,0.478516,0.363695,0.7652
Modelo 7 — Reduzido Conservador,0.731116,0.413932,0.455100,0.313579,0.7577


In [7]:
c_threshold_with_f1_05 = threshold_comparison_c[threshold_comparison_c["F1-score"]>=0.5]

c_best_thresholds_accuracy = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

c_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
5,Modelo 3 — Faixa Salarial,Random Forest Balanced,0.650,0.868,0.638,0.423,0.508,0.783
6,Modelo 5 — Estabilidade e Benefícios,XGBoost,0.325,0.857,0.567,0.479,0.519,0.795
1,Modelo 6 — Perfil Pessoal,Gradient Boosting Balanced,0.650,0.853,0.550,0.465,0.504,0.795
2,Modelo 3 — Faixa Salarial,Logistic Regression,0.300,0.848,0.529,0.507,0.518,0.796
3,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.848,0.529,0.521,0.525,0.796
7,Modelo 5 — Estabilidade e Benefícios,XGBoost Balanced,0.650,0.846,0.522,0.493,0.507,0.792
0,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.325,0.844,0.514,0.535,0.524,0.815
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.828,0.469,0.535,0.500,0.799


In [8]:
c_best_thresholds_precision = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

c_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
5,Modelo 3 — Faixa Salarial,Random Forest Balanced,0.650,0.868,0.638,0.423,0.508,0.783
6,Modelo 5 — Estabilidade e Benefícios,XGBoost,0.325,0.857,0.567,0.479,0.519,0.795
1,Modelo 6 — Perfil Pessoal,Gradient Boosting Balanced,0.650,0.853,0.550,0.465,0.504,0.795
2,Modelo 3 — Faixa Salarial,Logistic Regression,0.300,0.848,0.529,0.507,0.518,0.796
3,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.848,0.529,0.521,0.525,0.796
7,Modelo 5 — Estabilidade e Benefícios,XGBoost Balanced,0.650,0.846,0.522,0.493,0.507,0.792
0,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.325,0.844,0.514,0.535,0.524,0.815
4,Modelo 2 — Nível Hierárquico,Random Forest,0.250,0.828,0.469,0.535,0.500,0.799


In [9]:
c_best_thresholds_recall = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

c_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
5,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.475,0.748,0.368,0.789,0.502,0.798
3,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.500,0.773,0.393,0.746,0.515,0.818
1,Modelo 6 — Perfil Pessoal,Gradient Boosting Balanced,0.450,0.780,0.397,0.704,0.508,0.795
2,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.810,0.441,0.690,0.538,0.819
4,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.200,0.785,0.402,0.690,0.508,0.802
0,Modelo 3 — Faixa Salarial,Gradient Boosting,0.200,0.800,0.422,0.648,0.511,0.797
6,Modelo 2 — Nível Hierárquico,XGBoost,0.200,0.800,0.422,0.648,0.511,0.804
7,Modelo 5 — Estabilidade e Benefícios,XGBoost Balanced,0.525,0.791,0.407,0.648,0.500,0.792


In [10]:
c_best_thresholds_auc = c_threshold_with_f1_05.loc[
    c_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

c_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
2,Modelo 2 — Nível Hierárquico,Logistic Regression,0.200,0.810,0.441,0.690,0.538,0.819
3,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.500,0.773,0.393,0.746,0.515,0.818
0,Modelo 2 — Nível Hierárquico,Gradient Boosting,0.225,0.805,0.426,0.606,0.500,0.815
6,Modelo 6 — Perfil Pessoal,XGBoost,0.200,0.805,0.427,0.620,0.506,0.814
7,Modelo 6 — Perfil Pessoal,XGBoost Balanced,0.600,0.832,0.481,0.521,0.500,0.810
1,Modelo 2 — Nível Hierárquico,Gradient Boosting Balanced,0.550,0.800,0.422,0.648,0.511,0.805
4,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.200,0.785,0.402,0.690,0.508,0.802
5,Modelo 2 — Nível Hierárquico,Random Forest Balanced,0.600,0.855,0.554,0.507,0.529,0.802


# Comparação dos Modelos Mistos

In [11]:
general_comparison_mix, threshold_comparison_mix, confusion_results_mix, trained_models_mix, interpretation_results_mix = run_model_comparison_mixed(
    df=df,
    models_dict=mixed_models_dict_c,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

general_comparison_mix.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
70,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.5,0.880,0.696,0.451,0.547,0.816,7,11,43
19,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.5,0.825,0.468,0.620,0.533,0.834,3,8,25
17,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.5,0.821,0.457,0.606,0.521,0.834,3,8,25
11,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.5,0.773,0.393,0.746,0.515,0.832,3,8,25
21,Modelo 3 — Rendimento Quantitativo,Logistic Regression Balanced,0.5,0.760,0.378,0.761,0.505,0.797,4,6,19
...,...,...,...,...,...,...,...,...,...,...,...
54,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Random Forest,0.5,0.841,1.000,0.014,0.028,0.796,3,8,24
14,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.5,0.841,1.000,0.014,0.028,0.809,3,8,25
44,Modelo 5 — Antiguidade Organizacional,Random Forest,0.5,0.839,0.000,0.000,0.000,0.760,5,6,20
4,Modelo 1 — Função Profissional Misto,Random Forest,0.5,0.839,0.000,0.000,0.000,0.771,3,7,26


In [12]:
mix_best_thresholds_by_f1 = threshold_comparison_mix.loc[
    threshold_comparison_mix.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_by_f1.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
74,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.375,0.868,0.594,0.577,0.586,0.816
14,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.275,0.850,0.529,0.634,0.577,0.833
75,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813
18,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.300,0.859,0.565,0.549,0.557,0.833
12,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.844,0.512,0.606,0.555,0.826
...,...,...,...,...,...,...,...,...
30,Modelo 4 — Experiência Profissional,Decision Tree,0.225,0.753,0.339,0.563,0.423,0.677
38,Modelo 4 — Experiência Profissional,XGBoost,0.325,0.819,0.433,0.408,0.420,0.752
21,Modelo 3 — Rendimento Quantitativo,Decision Tree Balanced,0.250,0.594,0.259,0.817,0.393,0.717
20,Modelo 3 — Rendimento Quantitativo,Decision Tree,0.200,0.755,0.327,0.493,0.393,0.733


In [13]:
mix_best_thresholds_by_model = threshold_comparison_mix.loc[
    threshold_comparison_mix.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
4,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.375,0.868,0.594,0.577,0.586,0.816
5,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.300,0.859,0.565,0.549,0.557,0.833
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.844,0.512,0.606,0.555,0.826
9,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.525,0.837,0.494,0.620,0.550,0.834
3,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.425,0.800,0.431,0.746,0.546,0.834
7,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.550,0.841,0.506,0.549,0.527,0.813
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809
1,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree,0.225,0.828,0.463,0.437,0.449,0.726


In [14]:
mix_threshold_mean = threshold_comparison_mix.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean()

mix_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 2 — Nível Hierárquico e Benefícios,0.769584,0.455542,0.530489,0.414347,0.8062
Modelo 8 — Integrado Multidimensional,0.777379,0.437395,0.485463,0.402137,0.7818
Modelo 6 — Perfil Pessoal e Condições de Trabalho,0.761100,0.447105,0.475105,0.373932,0.7831
Modelo 5 — Antiguidade Organizacional,0.757732,0.427895,0.479484,0.368779,0.7660
Modelo 4 — Experiência Profissional,0.746063,0.400911,0.481295,0.365505,0.7542
Modelo 1 — Função Profissional Misto,0.759468,0.433732,0.464305,0.365253,0.7646
Modelo 3 — Rendimento Quantitativo,0.756147,0.436532,0.468874,0.365000,0.7804
Modelo 7 — Reduzido Conservador Misto,0.740611,0.395611,0.443326,0.322732,0.7570


In [15]:
mix_threshold_with_f1_05 = threshold_comparison_mix[threshold_comparison_mix["F1-score"]>=0.5]

mix_best_thresholds_accuracy = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.500,0.880,0.696,0.451,0.547,0.816
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.375,0.871,0.635,0.465,0.537,0.826
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.868,0.623,0.465,0.532,0.833
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.625,0.866,0.625,0.423,0.504,0.813
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.859,0.571,0.507,0.537,0.834
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.650,0.855,0.556,0.493,0.522,0.834
4,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722


In [16]:
mix_best_thresholds_precision = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.500,0.880,0.696,0.451,0.547,0.816
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.400,0.871,0.652,0.423,0.513,0.826
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.425,0.868,0.644,0.408,0.500,0.833
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.625,0.866,0.625,0.423,0.504,0.813
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.859,0.571,0.507,0.537,0.834
2,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.600,0.855,0.559,0.465,0.508,0.797
4,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722


In [17]:
mix_best_thresholds_recall = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.400,0.773,0.396,0.775,0.524,0.834
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.751,0.369,0.775,0.500,0.832
6,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.450,0.771,0.391,0.761,0.517,0.787
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.375,0.771,0.388,0.732,0.507,0.834
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.803,0.431,0.704,0.535,0.833
5,Modelo 8 — Integrado Multidimensional,Random Forest,0.200,0.794,0.414,0.676,0.513,0.795
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.200,0.816,0.450,0.634,0.526,0.826
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.819,0.454,0.620,0.524,0.833
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.805,0.426,0.606,0.500,0.722


In [18]:
mix_best_thresholds_auc = mix_threshold_with_f1_05.loc[
    mix_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

mix_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.400,0.773,0.396,0.775,0.524,0.834
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.375,0.771,0.388,0.732,0.507,0.834
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.803,0.431,0.704,0.535,0.833
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.819,0.454,0.620,0.524,0.833
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.751,0.369,0.775,0.500,0.832
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.200,0.816,0.450,0.634,0.526,0.826
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.525,0.821,0.457,0.592,0.515,0.813
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.805,0.426,0.606,0.500,0.722


# Comparação de Modelos Categóricos + Mistos

In [19]:
c_models_results = [general_comparison_c, threshold_comparison_c]
mix_models_results = [general_comparison_mix, threshold_comparison_mix]

for result in c_models_results:
    result["type"] = "Categorical"

for result in mix_models_results:
    result["type"] = "Mixed" 

In [20]:
all_comparison = pd.concat(
    [general_comparison_c, general_comparison_mix],
    ignore_index=True
)

all_threshold = pd.concat(
    [threshold_comparison_c, threshold_comparison_mix],
    ignore_index=True
)

In [21]:
all_comparison_by_f1 = all_threshold.loc[
    all_threshold.groupby(["Variable_Set", "Model"])["F1-score"].idxmax()
].reset_index(drop=True)

all_comparison_by_f1.sort_values(by="F1-score", ascending=False).head(20)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
144,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.375,0.868,0.594,0.577,0.586,0.816,Mixed
34,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.275,0.850,0.529,0.634,0.577,0.833,Mixed
145,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813,Mixed
38,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.300,0.859,0.565,0.549,0.557,0.833,Mixed
32,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.844,0.512,0.606,0.555,0.826,Mixed
39,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.525,0.837,0.494,0.620,0.550,0.834,Mixed
33,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.425,0.800,0.431,0.746,0.546,0.834,Mixed
45,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.625,0.841,0.506,0.592,0.545,0.795,Categorical
35,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.600,0.821,0.461,0.662,0.543,0.832,Mixed
25,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.575,0.819,0.456,0.662,0.540,0.818,Categorical


In [22]:
all_comparison_by_model = all_threshold.loc[
    all_threshold.groupby(["Model"])["F1-score"].idxmax()
].reset_index(drop=True)

all_comparison_by_model.sort_values(by="F1-score", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
4,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.375,0.868,0.594,0.577,0.586,0.816,Mixed
5,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.625,0.844,0.511,0.648,0.571,0.813,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.300,0.859,0.565,0.549,0.557,0.833,Mixed
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.225,0.844,0.512,0.606,0.555,0.826,Mixed
9,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.525,0.837,0.494,0.620,0.550,0.834,Mixed
3,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.425,0.800,0.431,0.746,0.546,0.834,Mixed
7,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.525,0.807,0.438,0.690,0.536,0.798,Categorical
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722,Mixed
0,Modelo 5 — Estabilidade e Benefícios,Decision Tree,0.275,0.821,0.452,0.535,0.490,0.768,Categorical


In [23]:
all_comparison_threshold_mean = all_threshold.groupby("Variable_Set")[[
    "Accuracy", "Precision", "Recall", "F1-score", "AUC"
]].mean()

all_comparison_threshold_mean.sort_values(by="F1-score", ascending=False)

,Accuracy,Precision,Recall,F1-score,AUC
Variable_Set,,,,,
Modelo 2 — Nível Hierárquico e Benefícios,0.769584,0.455542,0.530489,0.414347,0.8062
Modelo 3 — Faixa Salarial,0.751916,0.454121,0.534874,0.404879,0.7874
Modelo 8 — Integrado Multidimensional,0.777379,0.437395,0.485463,0.402137,0.7818
Modelo 2 — Nível Hierárquico,0.742158,0.438958,0.527747,0.384132,0.7901
Modelo 6 — Perfil Pessoal,0.756516,0.435084,0.501426,0.382642,0.7898
Modelo 6 — Perfil Pessoal e Condições de Trabalho,0.761100,0.447105,0.475105,0.373932,0.7831
Modelo 5 — Estabilidade e Benefícios,0.750211,0.439300,0.508663,0.371595,0.7893
Modelo 5 — Antiguidade Organizacional,0.757732,0.427895,0.479484,0.368779,0.7660
Modelo 4 — Trajetória Organizacional,0.745668,0.400258,0.487100,0.367795,0.7583


In [24]:
all_threshold_with_f1_05 = all_threshold[all_threshold["F1-score"]>=0.5]

all_best_thresholds_accuracy = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Accuracy"].idxmax()
].reset_index(drop=True)

all_best_thresholds_accuracy.sort_values(by="Accuracy", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.500,0.880,0.696,0.451,0.547,0.816,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.375,0.871,0.635,0.465,0.537,0.826,Mixed
6,Modelo 3 — Faixa Salarial,Random Forest Balanced,0.650,0.868,0.638,0.423,0.508,0.783,Categorical
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.400,0.868,0.623,0.465,0.532,0.833,Mixed
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.859,0.571,0.507,0.537,0.834,Mixed
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.650,0.855,0.556,0.493,0.522,0.834,Mixed
4,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.848,0.529,0.521,0.525,0.796,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722,Mixed


In [25]:
all_best_thresholds_precision = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Precision"].idxmax()
].reset_index(drop=True)

all_best_thresholds_precision.sort_values(by="Precision", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
3,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.500,0.880,0.696,0.451,0.547,0.816,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.400,0.871,0.652,0.423,0.513,0.826,Mixed
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.425,0.868,0.644,0.408,0.500,0.833,Mixed
6,Modelo 3 — Faixa Salarial,Random Forest Balanced,0.650,0.868,0.638,0.423,0.508,0.783,Categorical
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.650,0.859,0.571,0.507,0.537,0.834,Mixed
2,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.600,0.855,0.559,0.465,0.508,0.797,Mixed
4,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.650,0.848,0.529,0.521,0.525,0.796,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.650,0.821,0.455,0.563,0.503,0.722,Mixed


In [26]:
all_best_thresholds_recall = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["Recall"].idxmax()
].reset_index(drop=True)

all_best_thresholds_recall.sort_values(by="Recall", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
6,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.475,0.748,0.368,0.789,0.502,0.798,Categorical
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.400,0.773,0.396,0.775,0.524,0.834,Mixed
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.751,0.369,0.775,0.500,0.832,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.375,0.771,0.388,0.732,0.507,0.834,Mixed
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.803,0.431,0.704,0.535,0.833,Mixed
5,Modelo 5 — Estabilidade e Benefícios,Random Forest,0.200,0.785,0.402,0.690,0.508,0.802,Categorical
1,Modelo 3 — Faixa Salarial,Gradient Boosting,0.200,0.800,0.422,0.648,0.511,0.797,Categorical
7,Modelo 2 — Nível Hierárquico,XGBoost,0.200,0.800,0.422,0.648,0.511,0.804,Categorical
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.805,0.426,0.606,0.500,0.722,Mixed


In [27]:
all_best_thresholds_auc = all_threshold_with_f1_05.loc[
    all_threshold_with_f1_05.groupby(["Model"])["AUC"].idxmax()
].reset_index(drop=True)

all_best_thresholds_auc.sort_values(by="AUC", ascending=False)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,type
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.400,0.773,0.396,0.775,0.524,0.834,Mixed
8,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.375,0.771,0.388,0.732,0.507,0.834,Mixed
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.200,0.803,0.431,0.704,0.535,0.833,Mixed
7,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost,0.200,0.819,0.454,0.620,0.524,0.833,Mixed
4,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.475,0.751,0.369,0.775,0.500,0.832,Mixed
1,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting,0.200,0.816,0.450,0.634,0.526,0.826,Mixed
6,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest Balanced,0.525,0.821,0.457,0.592,0.515,0.813,Mixed
5,Modelo 2 — Nível Hierárquico e Benefícios,Random Forest,0.275,0.859,0.576,0.479,0.523,0.809,Mixed
0,Modelo 2 — Nível Hierárquico e Benefícios,Decision Tree Balanced,0.600,0.805,0.426,0.606,0.500,0.722,Mixed


# Cross-Validation

In [28]:
cv_comparison = run_cross_validation_mixed(df=df, models_dict=all_models_dict_c, estimators_dict=estimators_dict, target="AttritionFlag")

cv_comparison.sort_values("F1_Mean", ascending=False)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.780,0.042,0.408,0.060,0.780,0.087,0.535,0.071,0.839,0.057,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.816,0.031,0.448,0.070,0.590,0.087,0.509,0.075,0.811,0.050,7,11,43
147,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.814,0.022,0.444,0.053,0.590,0.099,0.504,0.060,0.808,0.049,7,11,43
37,Modelo 4 — Trajetória Organizacional,Gradient Boosting Balanced,0.768,0.041,0.384,0.064,0.687,0.093,0.490,0.068,0.776,0.046,0,8,22
145,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.796,0.024,0.412,0.048,0.607,0.080,0.489,0.053,0.806,0.040,7,11,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,Modelo 2 — Nível Hierárquico,Random Forest,0.841,0.005,0.300,0.458,0.013,0.019,0.024,0.037,0.762,0.052,0,8,22
64,Modelo 7 — Reduzido Conservador,Random Forest,0.839,0.003,0.200,0.400,0.008,0.017,0.016,0.032,0.735,0.047,0,7,18
134,Modelo 7 — Reduzido Conservador Misto,Random Forest,0.839,0.004,0.100,0.300,0.004,0.012,0.008,0.024,0.732,0.059,3,6,18
4,Modelo 1 — Função Profissional,Random Forest,0.839,0.003,0.000,0.000,0.000,0.000,0.000,0.000,0.770,0.043,0,8,26


In [29]:
best_cv_result = cv_comparison.loc[
    cv_comparison.groupby(["Variable_Set", "Model"])["F1_Mean"].idxmax()
].reset_index(drop=True)

best_cv_result.sort_values(by="F1_Mean", ascending=False).head(20)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
145,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.780,0.042,0.408,0.060,0.780,0.087,0.535,0.071,0.839,0.057,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.816,0.031,0.448,0.070,0.590,0.087,0.509,0.075,0.811,0.050,7,11,43
143,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.814,0.022,0.444,0.053,0.590,0.099,0.504,0.060,0.808,0.049,7,11,43
73,Modelo 4 — Trajetória Organizacional,Gradient Boosting Balanced,0.768,0.041,0.384,0.064,0.687,0.093,0.490,0.068,0.776,0.046,0,8,22
147,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.796,0.024,0.412,0.048,0.607,0.080,0.489,0.053,0.806,0.040,7,11,43
35,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.750,0.028,0.363,0.038,0.725,0.085,0.483,0.048,0.814,0.048,3,8,25
144,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.869,0.021,0.667,0.109,0.384,0.091,0.483,0.094,0.844,0.052,7,11,43
79,Modelo 4 — Trajetória Organizacional,XGBoost Balanced,0.763,0.042,0.376,0.059,0.661,0.080,0.476,0.054,0.775,0.044,0,8,22
105,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.740,0.029,0.354,0.033,0.730,0.040,0.476,0.036,0.795,0.049,0,9,24
65,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.737,0.048,0.353,0.050,0.729,0.076,0.474,0.057,0.789,0.051,4,6,19


In [30]:
top_combinations = (
    cv_comparison
    .sort_values("F1_Mean", ascending=False)
    .head(20)[["Variable_Set", "Model"]]
    .to_dict("records")
)

tuning_results_df, best_models = tune_hyperparameters_top_combinations(
    df=df,
    models_dict=all_models_dict_c,
    estimators_dict=estimators_dict,
    param_distributions_dict=param_distributions_dict,
    top_combinations=top_combinations,
    target="AttritionFlag",
    n_iter=30,
    n_splits=10,
    scoring="f1"
)

tuning_results_df.sort_values("Best_Score", ascending=False)

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 10 is smaller than n_iter=30. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1

,Variable_Set,Model,Best_Score,Scoring,Best_Params,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.532,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
2,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.532,f1,"{'classifier__subsample': 0.7, 'classifier__n_...",7,11,43
1,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.525,f1,"{'classifier__subsample': 0.9, 'classifier__sc...",7,11,43
4,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.511,f1,"{'classifier__n_estimators': 200, 'classifier_...",7,11,43
5,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.499,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
11,Modelo 3 — Rendimento Quantitativo,Random Forest Balanced,0.498,f1,"{'classifier__n_estimators': 200, 'classifier_...",4,6,19
13,Modelo 8 — Integrado Multidimensional,XGBoost,0.495,f1,"{'classifier__subsample': 1.0, 'classifier__re...",7,11,43
6,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.484,f1,"{'classifier__solver': 'liblinear', 'classifie...",3,8,25
16,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.480,f1,"{'classifier__n_estimators': 200, 'classifier_...",0,9,24
8,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.480,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,9,24


In [31]:
base_top_results = (
    cv_comparison
    .sort_values("F1_Mean", ascending=False)
    .head(20)
    [["Variable_Set", "Model", "F1_Mean", "F1_Std", "AUC_Mean"]]
)

optimized_results = tuning_results_df[["Variable_Set", "Model", "Best_Score", "Best_Params"]]

comparison_base_vs_optimized = base_top_results.merge(optimized_results, on=["Variable_Set", "Model"], how="left")

comparison_base_vs_optimized["F1_Improvement"] = (comparison_base_vs_optimized["Best_Score"] - comparison_base_vs_optimized["F1_Mean"])

comparison_base_vs_optimized.sort_values(by="Best_Score", ascending=False)

,Variable_Set,Model,F1_Mean,F1_Std,AUC_Mean,Best_Score,Best_Params,F1_Improvement
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.535,0.071,0.839,0.532,"{'classifier__solver': 'liblinear', 'classifie...",-0.003
2,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.504,0.060,0.808,0.532,"{'classifier__subsample': 0.7, 'classifier__n_...",0.028
1,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.509,0.075,0.811,0.525,"{'classifier__subsample': 0.9, 'classifier__sc...",0.016
4,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.489,0.053,0.806,0.511,"{'classifier__n_estimators': 200, 'classifier_...",0.022
5,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.483,0.094,0.844,0.499,"{'classifier__solver': 'liblinear', 'classifie...",0.016
11,Modelo 3 — Rendimento Quantitativo,Random Forest Balanced,0.472,0.051,0.774,0.498,"{'classifier__n_estimators': 200, 'classifier_...",0.026
13,Modelo 8 — Integrado Multidimensional,XGBoost,0.471,0.068,0.817,0.495,"{'classifier__subsample': 1.0, 'classifier__re...",0.024
6,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.483,0.048,0.814,0.484,"{'classifier__solver': 'liblinear', 'classifie...",0.001
16,Modelo 5 — Estabilidade e Benefícios,Random Forest Balanced,0.469,0.052,0.775,0.480,"{'classifier__n_estimators': 200, 'classifier_...",0.011
8,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.476,0.036,0.795,0.480,"{'classifier__solver': 'liblinear', 'classifie...",0.004


In [32]:
top_20_best_models = {
    (row["Variable_Set"], row["Model"]): best_models[(row["Variable_Set"], row["Model"])]
    for _, row in (
        tuning_results_df
        .sort_values("Best_Score", ascending=False)
        .iterrows()
    )
}

threshold_comparison_opt, best_thresholds_opt, confusion_results_opt, fitted_models_opt = evaluate_thresholds_optimized_models(
    df=df,
    models_dict=all_models_dict_c,
    best_models=top_20_best_models,
    estimators_dict=estimators_dict,
    target="AttritionFlag"
)

best_thresholds_opt

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
15,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.866,0.583,0.592,0.587,0.814
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.871,0.609,0.549,0.578,0.831
4,Modelo 2 — Nível Hierárquico e Benefícios,XGBoost Balanced,0.60,0.857,0.554,0.577,0.566,0.831
16,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.67,0.841,0.506,0.620,0.557,0.810
13,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.65,0.841,0.506,0.563,0.533,0.802
11,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.63,0.832,0.483,0.592,0.532,0.801
2,Modelo 2 — Nível Hierárquico e Benefícios,Gradient Boosting Balanced,0.45,0.830,0.477,0.577,0.522,0.810
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.60,0.821,0.457,0.606,0.521,0.798
12,Modelo 6 — Perfil Pessoal,XGBoost Balanced,0.40,0.762,0.384,0.789,0.516,0.807
5,Modelo 3 — Rendimento Quantitativo,Random Forest Balanced,0.46,0.807,0.430,0.606,0.503,0.790
